# 03 — Semantic auto-routing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenTracy/OpenTracy/blob/main/notebooks/03_semantic_routing.ipynb)

The auto-router turns **"I have thirteen models I could call"** into **"call the right one for this specific prompt"** — without you writing any rules.

How it works in one sentence: every prompt is embedded, assigned to one of 100 pre-trained semantic clusters, and scored against per-model error profiles. The router picks the model that minimizes `expected_error + λ · cost`.

In this notebook:

1. Load the pre-trained router (downloads ~100 MB once)
2. Route a handful of prompts — see which model gets picked and why
3. Tune the cost-quality tradeoff with `cost_weight`
4. Restrict the candidate pool
5. Combine routing with completion for a cost-optimizing client

> **Runtime:** CPU is fine. No API key needed for routing decisions. The final cell calls a real LLM — set `OPENAI_API_KEY` if you want to run it.

## Install

In [1]:
!pip install opentracy -q

## 1. Load the router

First run downloads `weights-default` from HuggingFace (~100 MB) and caches it under `~/.local/share/opentracy/`. Subsequent loads take ~2 seconds.

In [2]:
# ---- OpenTracy platform hook-up (optional but usually what you want) ----
# By default, opentracy.completion() and opentracy.Router go DIRECT to the
# provider (OpenAI, Anthropic, etc.). That means traces, cost accounting,
# routing decisions, and the distillation loop will NOT see your calls.
#
# To make every call pass through the running OpenTracy engine so it
# shows up in the dashboard, point OPENTRACY_ENGINE_URL at the ENGINE's
# HTTP API — default port **8080**. The dashboard UI on **:3000** is a
# different service (nginx) and will return 405 for POST /v1/chat/completions,
# so do NOT use that URL here.
#
# Provider API keys are configured once via the UI (Settings → API Keys)
# or POST /v1/secrets/<provider>; the engine reads them from
# ~/.opentracy/secrets.json inside its container on every call.

import os
os.environ["OPENTRACY_ENGINE_URL"] = "http://3.227.3.81:8080"  # engine API (:8080). Dashboard UI is on :3000 — NOT interchangeable.


In [3]:
import opentracy as ot

router = ot.load_router(cost_weight=0.5, verbose=True)

Using bundled weights at /home/ubuntu/lunar-router/opentracy/_bundled_weights/weights-mmlu-v1
Loading Go engine with weights: /home/ubuntu/lunar-router/opentracy/_bundled_weights/weights-mmlu-v1
Go engine ready! Models: 10, Clusters: 100, Embedder: True


The router returned wraps three things:

- a **sentence embedder** (bundled ONNX MiniLM-L6-v2 → 384-d vectors)
- a **cluster assigner** — 100 pre-trained semantic clusters with human-readable names
- **per-model error profiles** — Ψ vectors of length 100, one per model

`cost_weight` (λ = 0.5) is the knob you'll actually touch. `0.0` = quality-first (ignore cost); `1.0+` = cheap-first (escalate only on hard prompts).

## 2. Route some prompts

Let's see the router in action across a mix of difficulties.

In [4]:
prompts = [
    "What is the capital of France?",
    "Prove the square root of 2 is irrational.",
    "Write a Python function that reverses a linked list in place.",
    "Write a haiku about a lonely lighthouse.",
    "Summarize War and Peace in three paragraphs.",
    "Explain the CAP theorem to a junior engineer.",
    "Translate to French: 'Good morning, how are you?'",
    "Given this stack trace, find the bug: NullPointerException at UserService.java:42",
]

for p in prompts:
    d = router.route(p)
    print(f"[{d.selected_model:<28}] cluster={d.cluster_id:>3}  err={d.expected_error:.3f}  {p[:55]}")

[ministral-3b-latest         ] cluster= 84  err=0.000  What is the capital of France?
[gpt-4o                      ] cluster= 47  err=0.609  Prove the square root of 2 is irrational.
[gpt-4o                      ] cluster= 88  err=0.000  Write a Python function that reverses a linked list in 
[gpt-4o                      ] cluster= 87  err=0.212  Write a haiku about a lonely lighthouse.
[ministral-3b-latest         ] cluster= 16  err=0.000  Summarize War and Peace in three paragraphs.
[gpt-4o                      ] cluster= 93  err=0.111  Explain the CAP theorem to a junior engineer.
[ministral-3b-latest         ] cluster= 64  err=0.000  Translate to French: 'Good morning, how are you?'
[gpt-4o                      ] cluster= 92  err=0.212  Given this stack trace, find the bug: NullPointerExcept


Look at the picks. Easy trivia lands on cheap small models; math proofs and code problems escalate to strong ones; creative writing sits in the middle. **You didn't write any rules** — the router learned this from the benchmark the weights were trained on.

## 3. Full decision breakdown

Every route returns a `RoutingDecision` with the reasoning exposed.

In [5]:
d = router.route("Write a Python function that reverses a linked list.")

print(f"selected_model:        {d.selected_model}")
print(f"cluster_id:            {d.cluster_id}")
print(f"expected_error:        {d.expected_error:.4f}")
print(f"cost_adjusted_score:   {d.cost_adjusted_score:.4f}")
print()
print("all_scores (lower = better):")
for model, score in sorted(d.all_scores.items(), key=lambda x: x[1])[:5]:
    print(f"  {model:<28}  {score:.4f}")

selected_model:        gpt-4o
cluster_id:            88
expected_error:        0.0000
cost_adjusted_score:   0.0031

all_scores (lower = better):
  gpt-4o                        0.0031
  pixtral-12b-2409              0.1429
  gpt-4o-mini                   0.1430
  gpt-4-turbo                   0.1529
  ministral-8b-latest           0.2858


`all_scores` is the per-candidate score of `error + λ · cost` — the router picks the argmin.

## 4. Tune the cost-quality dial

Route the same hard prompt at four different `cost_weight` values to feel the tradeoff.

In [6]:
hard_prompt = "Prove that there are infinitely many primes using the method of Euclid."

for lam in [0.0, 0.5, 1.0, 2.0]:
    d = router.route(hard_prompt, cost_weight_override=lam)
    print(f"λ={lam:>4}  →  {d.selected_model:<28} (err={d.expected_error:.3f})")

λ= 0.0  →  mistral-large-latest         (err=0.083)
λ= 0.5  →  mistral-small-latest         (err=0.083)
λ= 1.0  →  mistral-small-latest         (err=0.083)
λ= 2.0  →  mistral-small-latest         (err=0.083)


| λ | Behavior |
| --- | --- |
| `0.0` | Quality-first — pick the model with the lowest predicted error, ignore cost entirely |
| `0.5` | Balanced — a tiny error delta won't justify a 10× cost |
| `1.0` | Strongly prefer cheaper models, escalate only if they're demonstrably bad |
| `2.0+` | Aggressively cheap — escalate only on the worst prompts |

Try a few values on your own traffic. The right number depends on how much quality degradation you can tolerate.

## 5. Restrict the candidate pool

Sometimes you want to A/B between just two or three models, or a provider isn't available in a tenant. Constrain the pool with `allowed_models` at load time, or `available_models` per call.

In [7]:
# Only route among these three models globally
restricted = ot.load_router(
    cost_weight=0.5,
    allowed_models=["gpt-4o-mini", "ministral-3b-latest", "gpt-4o"],
    verbose=False,
)

for p in prompts[:4]:
    d = restricted.route(p)
    print(f"[{d.selected_model:<24}]  {p}")

[ministral-3b-latest     ]  What is the capital of France?
[gpt-4o                  ]  Prove the square root of 2 is irrational.
[gpt-4o                  ]  Write a Python function that reverses a linked list in place.
[gpt-4o                  ]  Write a haiku about a lonely lighthouse.


In [8]:
# Or override per call without reloading
d = router.route(
    "Write a haiku about autumn.",
    available_models=["gpt-4o-mini", "ministral-3b-latest"],
)
print(f"picked: {d.selected_model} (score={d.cost_adjusted_score:.4f})")

picked: gpt-4o-mini (score=0.2882)


## 6. Batch routing

If you're classifying a batch of prompts offline, `route_batch` embeds them all in one pass — much faster than looping.

In [9]:
decisions = router.route_batch(prompts)

from collections import Counter
distribution = Counter(d.selected_model for d in decisions)
print("Distribution across this batch:")
for model, count in distribution.most_common():
    print(f"  {model:<28}  {count}")

Distribution across this batch:
  gpt-4o                        5
  ministral-3b-latest           3


## 7. Combine: route + complete

The smallest useful cost-optimizing client — route first, then call the picked model.

In [10]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

def smart_completion(prompt: str) -> str:
    d = router.route(prompt)
    provider_prefix = "openai/" if d.selected_model.startswith("gpt") or d.selected_model.startswith("o1") else ""
    resp = ot.completion(
        model=f"{provider_prefix}{d.selected_model}",
        messages=[{"role": "user", "content": prompt}],
    )
    print(f"  [{d.selected_model}] cluster={d.cluster_id} err={d.expected_error:.2f} cost=${resp._cost:.6f}")
    return resp.choices[0].message.content

answer = smart_completion("What is 2 + 2?")
print("→", answer[:80])
print()

answer = smart_completion("Write a Python one-liner that returns the longest word in a string.")
print("→", answer[:100])

  [gpt-4o] cluster=88 err=0.00 cost=$0.000117
→ 2 + 2 equals 4.

  [gpt-4o] cluster=88 err=0.00 cost=$0.001055
→ You can achieve this by using the `max()` function with a key that is the `len` function, all in a o


## What you just saw

- ✅ **One router, 100 semantic clusters, 13 models** — all pre-trained
- ✅ **`cost_weight` as the single knob** for the quality ↔ cost tradeoff
- ✅ **Candidate restriction** via `allowed_models` / `available_models`
- ✅ **Batch routing** for offline classification
- ✅ **Route + complete** as a drop-in replacement for a single provider call

## When this isn't enough

- **Hard policy constraints** ("never Anthropic for PII prompts") — combine with the rule-based `Router` class
- **Your prompts don't cluster well** — retrain weights on your own traffic with `opentracy.training.full_training_pipeline`

## Next

- **04 — Ticket classifier (end-to-end)** → a concrete 50-line app with before/after cost
- **05 — Distillation** → the compounding payoff (needs a GPU)